
# Pipeline 1/2 — Nettoyage des données

Premier notebook du pipeline : charger les onze fichiers bruts, auditer leur qualité, appliquer les corrections, et produire le jeu de données propre que le notebook d'entraînement consommera.

Chaque correction appliquée ici est **justifiée par une mesure** — l'audit complet, les cinq anomalies non annoncées par l'énoncé et le journal des choix sont dans le dossier technique (`soutenance_cisia.ipynb`). Ce notebook en est la version pipeline : lisible de bout en bout, sans détour.


In [1]:

import sys, warnings
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src import config as cfg, loading, audit, clean, assemble

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 140)


## 1. Chargement des données brutes

In [2]:

tables = loading.load_raw()
loading.table_overview(tables)


,lignes,colonnes,cle_primaire,unicite_cle
table,,,,
patients,620,11,PatientID,True
historique,759,6,EvenementID,True
sejours,878,10,SejourID,True
diagnostics,2062,5,DiagnosticID,False
actes,1254,6,ActeID,True
biologies,3721,8,BiologieID,False
signes_vitaux,24916,9,ConstanteID,True
medications,3728,9,PrescriptionID,True
comptes_rendus,1144,5,CompteRenduID,True



## 2. Audit — ce que les données contiennent vraiment

Quatre contrôles résumés (le dossier technique en contient neuf) : les manquants réels — sentinelles textuelles comprises —, l'intégrité référentielle, la cohérence de la durée de séjour, et la cible par mode de sortie.


In [3]:

audit.missing_report(tables).head(14)


,table,colonne,nuls,sentinelles_texte,total,taux
0,signes_vitaux,FrequenceRespiratoire,2021,0,2021,0.081113
1,signes_vitaux,Temperature,1673,0,1673,0.067146
2,signes_vitaux,TensionDiastolique,1471,0,1471,0.059038
3,signes_vitaux,SpO2,1428,0,1428,0.057313
4,signes_vitaux,TensionSystolique,1275,0,1275,0.051172
5,signes_vitaux,FrequenceCardiaque,1245,0,1245,0.049968
6,medications,DateFin,777,0,777,0.208423
7,historique,DureeJours,326,0,326,0.429513
8,biologies,Valeur,299,0,299,0.080355
9,patients,PersonneAPrevenir,0,105,105,0.169355


In [4]:

audit.referential_integrity(tables)


,table,cle,reference,orphelins,parents_couverts,parents_total,couverture
0,historique,PatientID,patients.PatientID,0,411,620,0.662903
1,sejours,PatientID,patients.PatientID,0,620,620,1.000000
2,diagnostics,SejourID,sejours.SejourID,0,878,878,1.000000
3,actes,SejourID,sejours.SejourID,0,649,878,0.739180
4,biologies,SejourID,sejours.SejourID,73,879,878,1.001139
5,signes_vitaux,SejourID,sejours.SejourID,0,878,878,1.000000
6,medications,SejourID,sejours.SejourID,0,878,878,1.000000
7,comptes_rendus,SejourID,sejours.SejourID,0,878,878,1.000000
8,objets_connectes,PatientID,patients.PatientID,125,184,620,0.296774


In [5]:

audit.stay_internal_coherence(tables)


,min,max,negatives,sup_28j
source,,,,
colonne DureeSejour,-3.0,99.0,20,15
recalcul DateSortie - DateAdmission,0.0,28.0,0,0


In [6]:

audit.target_by_discharge_mode(tables)


,sejours,readmissions,taux
ModeSortie,,,
Transfert,144,28,0.194444
Domicile,480,89,0.185417
Deces,92,17,0.184783
HAD,75,12,0.160000
EHPAD,87,12,0.137931



Quatre constats guident le nettoyage :

1. les manquants de `patients.csv` sont encodés en texte (« Non renseigné ») — invisibles à `isna()` ;
2. 73 lignes de biologie sont orphelines (séjour inexistant) ;
3. la colonne `DureeSejour` est corrompue (valeurs négatives), mais les dates dont elle dérive sont saines ;
4. les séjours terminés par un décès portent un taux de réadmission impossible (18,5 %) — ils sortent de la population d'étude.



## 3. Nettoyage et assemblage

Les corrections : normalisation des sentinelles, recalcul de la durée depuis les dates, exclusion des décès, harmonisation des unités biologiques (créatinine ×8,84 vers µmol/L, hémoglobine ×10 vers g/L), mise à l'écart des valeurs physiologiquement impossibles, recalage temporel intra-séjour.

L'assemblage produit **une ligne par séjour**, à partir des dix blocs de variables de l'étage A — les sources disponibles au moment de la sortie. Les objets connectés (postérieurs à la sortie) en sont exclus par construction.


In [7]:

population = clean.build_population(tables)
X, y, groupes = assemble.build_dataset(tables, stages=("A",),
                                       population=population)


  bloc demographie        etage A   14 variables   786 sejours renseignes
  bloc sejour             etage A   12 variables   786 sejours renseignes
  bloc diagnostics        etage A    7 variables   786 sejours renseignes
  bloc actes              etage A    4 variables   786 sejours renseignes
  bloc biologie           etage A   27 variables   786 sejours renseignes


  bloc constantes         etage A   44 variables   786 sejours renseignes


  bloc medications        etage A    5 variables   786 sejours renseignes
  bloc texte              etage A    8 variables   786 sejours renseignes


  bloc historique         etage A    7 variables   786 sejours renseignes
  bloc territoire         etage A    4 variables   786 sejours renseignes

  total : 786 sejours x 132 variables
  cible : 141 readmissions (17.9%)
  groupes : 575 patients


## 4. Écriture du jeu de données propre

In [8]:

sortie = Path("outputs")
sortie.mkdir(exist_ok=True)

donnees = X.copy()
for c in donnees.select_dtypes("category").columns:
    donnees[c] = donnees[c].astype(str)      # compatibilité parquet
donnees[cfg.TARGET] = y
donnees["PatientID"] = groupes
donnees.to_parquet(sortie / "donnees_propres.parquet")

colonnes_cat = [c for c in X.columns if str(X[c].dtype) == "category"]
(sortie / "colonnes_categorielles.txt").write_text(
    "\n".join(colonnes_cat), encoding="utf-8")

print(f"écrit : outputs/donnees_propres.parquet — "
      f"{donnees.shape[0]} séjours × {X.shape[1]} variables")
print(f"cible : {int(y.sum())} réadmissions ({y.mean():.1%})")


écrit : outputs/donnees_propres.parquet — 786 séjours × 132 variables
cible : 141 réadmissions (17.9%)



---

**Sortie du notebook** : `outputs/donnees_propres.parquet` — 786 séjours, 133 variables, la cible et l'identifiant patient (nécessaire au découpage sans fuite du notebook suivant).

→ Suite du pipeline : `02_entrainement_modele.ipynb`
